## CosmicFish v1.0

In [1]:
%matplotlib inline

In [2]:
#Importing main module
from cosmicfishpie.fishermatrix import cosmicfish
import numpy as np
import os

In [3]:
envkey = 'OMP_NUM_THREADS'
# Set this environment variable to the number of available cores in your machine, 
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(8)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

The value of OMP_NUM_THREADS is:  None
The value of OMP_NUM_THREADS is:  8


## CosmicFish in all modes

In [4]:
lss_fish_dic = dict()

## EXT

In [5]:
external = {'directory': '/media/santiago/SecondDisk/Euclid-Files/w0wafiles/default_camb_euclid_w0wa_HP/',  ## Files should be in the input4cast format
            'paramnames': ['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0','wa'],  ## Certain paramnames like Omegam and h are obligatory
            'folder_paramnames': ['Om', 'Ob', 'h', 'ns', 's8', 'w0', 'wa'],   ## Folder paramnames can have different names as paramnames
            'file_prefixes' : ['background_Hz','D_Growth-zk',  
                               'f_GrowthRate-zk', 'Plin-zk',    ## Names of cosmological quantity files can be specified here
                               'Pnonlin-zk', 'sigma8-z'],
            'k-units' : 'h/Mpc',   ## Units of the external input files
            'r-units' : 'Mpc',      
            'eps_values': [0.00625, 0.01, 0.0125, 0.01875, 0.025, 0.03, 0.0375, 0.05, 0.10]   ## Epsilon parameter variations at which files were computed
            } 

fiducial = {"Omegam":0.32,
            "Omegab":0.05,
            "h":0.67,
            "ns":0.96,          ## Fiducial values of the cosmological parameters
            "sigma8":0.815584,
            "w0":-1.0,
            "wa":0.
            }
## Fiducial values of the nuisance parameters are set by default when specifying the survey below. Can be added also manually.
freepars = {"Omegam":0.01,
            "Omegab":0.01 ,
            "h":0.01,           
            "ns":0.01,        ## If derivatives are calculated with 3PT, this sets the epsilon step size, per parameter. 
            "sigma8":0.01,      ## Should match one of the epsilons available above
            "w0":0.01,
            "wa":0.01
            } 

#specifications = ['ISTF-Optimistic', 'DR1-forza']
specifications = ['ISTF-Pessimistic', 'DR1-debole']
cosmoFM = dict()
feed=0
for specif in specifications:
    options = {
           #'derivatives': '4PT_FWD',      ## Derivative option: 3PT or SteM
           'derivatives': '3PT',      ## Derivative option: 3PT or SteM
           'accuracy': 1,
           'feedback': feed,
           'code': 'external',
           'outroot': 'w0waCDM-ext-Euclid-{:s}'.format(specif),    #String attached to all the results files
           'results_dir' :  './results/',
           'specs_dir' : '../survey_specifications/',     
           'survey_name_photo': 'Euclid-Photometric-'+specif,
           'survey_name_spectro': 'Euclid-Spectroscopic-'+specif,
           'cosmo_model' : 'w0waCDM',
           'activateMG': False}
    feed += 1
    observables = [['GCsp'], ['WL', 'GCph']]
    for obse in observables:              
    #Observables for which to compute the Fisher
        savekey = specif+str(obse) 
        cosmoFM[savekey] = cosmicfish.FisherMatrix(fiducialpars=fiducial,    #Pass the above dictionaries to cosmoFM, the main cosmicfish class
                                  freepars=freepars,
                                  options=options, 
                                  observables=obse, 
                                  extfiles=external, 
                                  cosmoModel=options['cosmo_model']
                                  )
                                
        lss_fish_dic[options['outroot']] = cosmoFM[savekey].compute()    # Compute the Fisher Matrix

**************************************************************
   _____               _     _____     __  
  / ___/__  ___ __ _  (_)___/ __(_)__ / /  
 / /__/ _ \(_-</  ' \/ / __/ _// (_-</ _ \ 
 \___/\___/___/_/_/_/_/\__/_/ /_/___/_//_/ 

**************************************************************
 This is the new Python version of the CosmicFish code.
**************************************************************

  -> Using input files for cosmology observables: /media/santiago/SecondDisk/Euclid-Files/w0wafiles/default_camb_euclid_w0wa_HP/
Custom fiducial parameters loaded
Entering Cov gg term
Computing derivatives of Galaxy Clustering Spectro
>> Computing Derivs >>
ððð Obtaining analytical derivative for parameter:  Ps_1
ððð Obtaining analytical derivative for parameter:  Ps_2
ððð Obtaining analytical derivative for parameter:  Ps_3
ððð Obtaining analytical derivative for parameter:  Ps_4

In class: SpectroDerivs  -->> Derivatives computed in   36.20 s
Redshift binned Fisher sha

KeyboardInterrupt: 

# Plot the resulting Fisher matrices

In [6]:
%matplotlib inline
from cosmicfishpie.analysis import fisher_plotting as fpp
from cosmicfishpie.analysis import fisher_matrix as fm
from cosmicfishpie.analysis import fisher_operations as fo
import glob
import numpy as np
import os
import matplotlib.pyplot as plt
plt.style.use('../scripts/plot-style.txt')
import seaborn as sns
snscolors=sns.color_palette("Set2")

In [7]:
filenames = glob.glob('./results/*CosmicFish*w0waCDM*ext-_*fishermatrix*.txt')
filenames

['./results/CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_WLGCph_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_GCsp_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_WLGCph_fishermatrix.txt']

In [8]:
filenames = [filenames[ii] for ii in [0, 1, 2, 3]]
filenames

['./results/CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_WLGCph_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_GCsp_fishermatrix.txt',
 './results/CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_WLGCph_fishermatrix.txt']

In [9]:
Fishes = [fm.fisher_matrix(file_name=file) for file in filenames]
for ffi in Fishes:
    print(ffi.name)
    print(ffi.get_param_names())
    #print(ffi.get_param_fiducial())

CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'lnbgs8_1', 'lnbgs8_2', 'lnbgs8_3', 'lnbgs8_4', 'Ps_1', 'Ps_2', 'Ps_3', 'Ps_4']
CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_WLGCph_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'b10', 'AIA', 'betaIA', 'etaIA']
CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_GCsp_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'lnbgs8_1', 'lnbgs8_2', 'lnbgs8_3', 'lnbgs8_4', 'Ps_1', 'Ps_2', 'Ps_3', 'Ps_4']
CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_WLGCph_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'AIA', 'betaIA', 'etaIA']


In [10]:
fishEucCombISTF = Fishes[0]+Fishes[1]
fishEucCombDR1 = Fishes[2]+Fishes[3]

Fishes.append(fishEucCombISTF)
Fishes.append(fishEucCombDR1)

In [11]:
for ffi in Fishes:
    print(ffi.name)
    print(ffi.get_param_names())

CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'lnbgs8_1', 'lnbgs8_2', 'lnbgs8_3', 'lnbgs8_4', 'Ps_1', 'Ps_2', 'Ps_3', 'Ps_4']
CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_WLGCph_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'b10', 'AIA', 'betaIA', 'etaIA']
CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_GCsp_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'lnbgs8_1', 'lnbgs8_2', 'lnbgs8_3', 'lnbgs8_4', 'Ps_1', 'Ps_2', 'Ps_3', 'Ps_4']
CosmicFish_v1.0_DR1s-w0waCDM-DR1-debole-ext-_WLGCph_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'AIA', 'betaIA', 'etaIA']
CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix_CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_WLGCph_fishermatrix
['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa', 'lnbgs8_1', '

In [12]:
plotFishnames = [
                 'CF-w0waCDM-ISTFpes-GCsp', 
                 'CF-w0waCDM-ISTFpes-3x2ph',
                 'CF-w0waCDM-DR1deb-GCsp', 
                 'CF-w0waCDM-DR1deb-3x2ph', 
                 'CF-w0waCDM-ISTFpes-GCsp+3x2ph',
                 'CF-w0waCDM-DR1deb-GCsp+3x2ph',
                 ]

In [13]:
for ii, fish in enumerate(Fishes):
    print("----")
    print("Old Fisher Name: ", fish.name)
    fish.name = plotFishnames[ii]
    print("New Fisher Name: ", fish.name)
    sigmas = fish.get_confidence_bounds()
    fidus = fish.get_param_fiducial()
    parnames = fish.get_param_names()
    fiww = fo.marginalise(fish, ['w0','wa'])
    deFoM = np.sqrt(fiww.determinant())
    print("Fisher DE FoM: ", deFoM)
    for ii, par in enumerate(parnames):
        print("Parameter {:s},  fiducial: {:.3f}, 1-sigma error: {:.4f}, percent error: {:.1f}%".format(
            par, fidus[ii], abs(sigmas[ii]), abs(100*sigmas[ii]/fidus[ii])))

----
Old Fisher Name:  CosmicFish_v1.0_DR1s-w0waCDM-ISTF-Pessimistic-ext-_GCsp_fishermatrix
New Fisher Name:  CF-w0waCDM-ISTFpes-GCsp
Fisher DE FoM:  82.38298504467227
Parameter Omegam,  fiducial: 0.320, 1-sigma error: 0.0141, percent error: 4.4%
Parameter Omegab,  fiducial: 0.050, 1-sigma error: 0.0021, percent error: 4.1%
Parameter h,  fiducial: 0.670, 1-sigma error: 0.0187, percent error: 2.8%
Parameter ns,  fiducial: 0.960, 1-sigma error: 0.0126, percent error: 1.3%
Parameter sigma8,  fiducial: 0.816, 1-sigma error: 0.0145, percent error: 1.8%
Parameter w0,  fiducial: -1.000, 1-sigma error: 0.1187, percent error: 11.9%
Parameter wa,  fiducial: 0.000, 1-sigma error: 0.3517, percent error: inf%
Parameter lnbgs8_1,  fiducial: -0.325, 1-sigma error: 0.0152, percent error: 4.7%
Parameter lnbgs8_2,  fiducial: -0.316, 1-sigma error: 0.0150, percent error: 4.7%
Parameter lnbgs8_3,  fiducial: -0.312, 1-sigma error: 0.0148, percent error: 4.8%
Parameter lnbgs8_4,  fiducial: -0.320, 1-sigma e

/tmp/ipykernel_2508/131750190.py:14: RuntimeWarning: divide by zero encountered in scalar divide
  par, fidus[ii], abs(sigmas[ii]), abs(100*sigmas[ii]/fidus[ii])))


In [14]:
plotPars = (Fishes[0].param_names)[:7]
print(plotPars)

['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa']


In [15]:
snscolors

[(0.4, 0.7607843137254902, 0.6470588235294118),
 (0.9882352941176471, 0.5529411764705883, 0.3843137254901961),
 (0.5529411764705883, 0.6274509803921569, 0.796078431372549),
 (0.9058823529411765, 0.5411764705882353, 0.7647058823529411),
 (0.6509803921568628, 0.8470588235294118, 0.32941176470588235),
 (1.0, 0.8509803921568627, 0.1843137254901961),
 (0.8980392156862745, 0.7686274509803922, 0.5803921568627451),
 (0.7019607843137254, 0.7019607843137254, 0.7019607843137254)]

In [16]:
for ii, ffi in enumerate(Fishes):
    print("i=", ii, ": ", ffi.name)

i= 0 :  CF-w0waCDM-ISTFpes-GCsp
i= 1 :  CF-w0waCDM-ISTFpes-3x2ph
i= 2 :  CF-w0waCDM-DR1deb-GCsp
i= 3 :  CF-w0waCDM-DR1deb-3x2ph
i= 4 :  CF-w0waCDM-ISTFpes-GCsp+3x2ph
i= 5 :  CF-w0waCDM-DR1deb-GCsp+3x2ph


In [17]:
whichFish = [0,1,2,3,4,5]
plot_options = {
          'fishers_list': [Fishes[ii] for ii in whichFish],
          'colors': [snscolors[ii] for ii in whichFish],
          'fish_labels' : [Fishes[ii].name for ii in whichFish],
          'plot_pars': plotPars,
          'axis_custom_factors': {'all': 3},  ## Axis limits cover 3-sigma bounds of first Fisher matrix
          'plot_method': 'Gaussian',
          'file_format': '.pdf',   ##file format for all the plots
          'outpath' : './plots/',  ## directory where to store the files, if non-existent, it will be created
          'outroot':'CF-WL+GCsp-pess'  ## file name root for all the plots, extra names can be added individually
               } 
fish_plotter = fpp.fisher_plotting(**plot_options)
fish_plotter.load_gaussians()
#fish_plotter.plot_fisher(filled=False, contour_args=[{'alpha':0.99, 'ls': '--'}, {'alpha':0.85, 'ls': '--'}])
fish_plotter.compare_errors(options={'ncol_legend': 2, 'yrang': [-500, 2000], 'compare_to_index':4})

./plots  exists already
Fisher matrix loaded, label name:  CF-w0waCDM-ISTFpes-GCsp
Fisher matrix loaded, label name:  CF-w0waCDM-ISTFpes-3x2ph
Fisher matrix loaded, label name:  CF-w0waCDM-DR1deb-GCsp
Fisher matrix loaded, label name:  CF-w0waCDM-DR1deb-3x2ph
Fisher matrix loaded, label name:  CF-w0waCDM-ISTFpes-GCsp+3x2ph
Fisher matrix loaded, label name:  CF-w0waCDM-DR1deb-GCsp+3x2ph
('Fishers names: ', ['CF-w0waCDM-ISTFpes-GCsp', 'CF-w0waCDM-ISTFpes-3x2ph', 'CF-w0waCDM-DR1deb-GCsp', 'CF-w0waCDM-DR1deb-3x2ph', 'CF-w0waCDM-ISTFpes-GCsp+3x2ph', 'CF-w0waCDM-DR1deb-GCsp+3x2ph'])
('parameters to plot: ', ['Omegam', 'Omegab', 'h', 'ns', 'sigma8', 'w0', 'wa'])
X tick labels ---> :   ['\\Omega_{{\\rm m}, 0}', '\\Omega_{{\\rm b}, 0}', 'h', 'n_{\\rm s}', '\\sigma_8', 'w_0', 'w_a']


In [18]:
fish_plotter.plot_fisher(filled=[True,True,False,False,True, False], 
                          contour_args=[{'alpha':0.85, 'ls': '-'},  
                                        {'alpha':0.85, 'ls': '-'},
                                        {'alpha':0.85, 'ls': '-'},
                                        {'alpha':0.85, 'ls': '-'},
                                        {'alpha':0.85, 'ls': '-'},
                                        {'alpha':0.85, 'ls': '-'}
                                        ])

Entering plotting routine
{'Omegam': [0.27766413526820655, 0.36233586473179347], 'Omegab': [0.04379742705862562, 0.05620257294137439], 'h': [0.613947880227574, 0.7260521197724261], 'ns': [0.9222780111013389, 0.997721988898661], 'sigma8': [0.7722288998459527, 0.8589391001540473], 'w0': [-1.3562459470034285, -0.6437540529965715], 'wa': [-1.055239549331938, 1.055239549331938], 'lnbgs8_1': [-0.3557704533590468, -0.2950695466409532], 'lnbgs8_2': [-0.3458261672109201, -0.2859878327890799], 'lnbgs8_3': [-0.34115336035201865, -0.28192063964798136], 'lnbgs8_4': [-0.34944420914310764, -0.2909117908568924], 'Ps_1': [-74.46885561772406, 74.46885561772406], 'Ps_2': [-74.90784311031905, 74.90784311031905], 'Ps_3': [-78.36819337389822, 78.36819337389822], 'Ps_4': [-79.53628804834784, 79.53628804834784]}


/media/team_workspaces/EC-THSWG/cosmicfishpie_split/cosmicfishpie/analysis/fisher_plotting.py:258: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  g.fig.savefig(
/opt/miniconda/lib/python3.11/site-packages/IPython/core/events.py:93: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  func(*args, **kwargs)
